# 14 - LLM Pre-training

**AI sin humo** - Notas personales para entender deep learning desde cero.

Llegamos al final de la serie. Acá hablamos de **cómo se entrenan los Large Language Models (LLMs)** como GPT, LLaMA, etc. Es básicamente lo que vimos en los notebooks anteriores, pero **a una escala enorme**.

---

## Contenido

1. [¿Qué es pre-training?](#pretraining)
2. [Datasets](#datasets)
3. [Tokenizers (BPE)](#tokenizers)
4. [Hiperparámetros de entrenamiento](#hparams)
5. [Scaling laws](#scaling)
6. [Lo que sigue: post-training](#posttraining)
7. [Resumen de la serie completa](#resumen)

---

<a id='pretraining'></a>
## 1. ¿Qué es pre-training?

Pre-training es el primer paso para crear un LLM. Consiste en entrenar un modelo Transformer (decoder-only, como vimos en el notebook 12) con una tarea muy simple pero a una escala masiva:

> **Dado un texto, predecir el siguiente token.**

Eso es todo. El modelo ve billones de tokens de texto y aprende a predecir qué viene después. De esta tarea aparentemente simple surgen capacidades increíbles:

- **Conocimiento del lenguaje**: gramática, sintaxis, semántica
- **Conocimiento del mundo**: hechos, relaciones, conceptos
- **Razonamiento**: lógica, matemáticas (hasta cierto punto)
- **Capacidad de seguir instrucciones**: surge de haber visto texto instructivo

### ¿Por qué funciona?

Para predecir bien el siguiente token en un texto complejo, el modelo **necesita entender** el contenido. Si lee un párrafo de física, para predecir la palabra siguiente necesita "entender" la física. Si lee código, necesita "entender" programación.

No es que el modelo realmente "entienda" en el sentido humano, pero desarrolla representaciones internas que capturan estructura, relaciones y conocimiento de una manera que le permite hacer predicciones extremadamente buenas.

### La escala

Los LLMs modernos se entrenan con:
- **Billones de tokens** (1-15 trillones de tokens)
- **Miles de millones de parámetros** (7B, 70B, 400B+)
- **Miles de GPUs** en paralelo
- **Semanas o meses** de entrenamiento continuo
- **Millones de dólares** en cómputo

---

<a id='datasets'></a>
## 2. Datasets

El dataset de pre-training es **enorme** y viene de múltiples fuentes:

| Fuente | Descripción | Proporción típica |
|--------|-------------|-------------------|
| **Web crawls** | CommonCrawl, refinado (FineWeb, etc.) | 60-80% |
| **Libros** | Books3, BookCorpus, Gutenberg | 5-10% |
| **Código** | GitHub, StackOverflow | 5-15% |
| **Papers académicos** | ArXiv, S2ORC | 2-5% |
| **Wikipedia** | Multi-idioma | 2-5% |
| **Conversaciones** | Reddit, foros | 2-5% |

### Filtrado y calidad

No todo lo que está en internet es útil. Se aplican filtros pesados:

- **Deduplicación**: eliminar textos repetidos (hay MUCHO contenido duplicado en la web)
- **Filtrado de calidad**: modelos clasificadores que distinguen texto de alta vs baja calidad
- **Filtrado de contenido tóxico/inapropiado**
- **Detección de idioma**: asegurar la distribución deseada de idiomas
- **Filtrado de datos personales** (PIIs)

La calidad del dataset es **tan importante o más** que el tamaño del modelo. Garbage in, garbage out.

---

<a id='tokenizers'></a>
## 3. Tokenizers (BPE)

En el notebook 12 usamos un tokenizer a nivel de caracteres. Los LLMs reales usan **BPE (Byte Pair Encoding)** o variantes.

### ¿Qué es BPE?

Es un algoritmo que aprende a dividir texto en "subpalabras" (subwords) de manera óptima:

1. Empezás con todos los bytes individuales como vocabulario base (256 tokens)
2. Contás qué par de tokens adyacentes aparece más frecuentemente en el corpus
3. Fusionás ese par en un nuevo token
4. Repetís hasta tener el vocabulario deseado (32K, 50K, 100K+ tokens)

**Ejemplo**: Si "th" y "e" aparecen mucho juntos, se crea el token "the". Si "ing" es frecuente, se crea ese token. Palabras comunes como "the", "and" terminan siendo un solo token. Palabras raras se descomponen en subpalabras.

### ¿Por qué no caracteres?

- Las secuencias serían **mucho más largas** (cada carácter = un token)
- Attention es $O(T^2)$, así que secuencias largas son caras
- BPE comprime: "understanding" puede ser 1-3 tokens en vez de 13 caracteres

### ¿Por qué no palabras completas?

- Vocabulario enorme e imposible de manejar
- Palabras nuevas o raras serían desconocidas (OOV)
- BPE puede representar **cualquier** texto combinando subpalabras

**Referencia**: [minBPE de Karpathy](https://github.com/karpathy/minbpe) es una implementación educativa excelente.

In [ ]:
# Simple BPE concept demo
from collections import Counter

def simple_bpe_step(tokens_list):
    """One step of BPE: find most frequent pair and merge it."""
    # Count all adjacent pairs
    pair_counts = Counter()
    for tokens in tokens_list:
        for i in range(len(tokens) - 1):
            pair_counts[(tokens[i], tokens[i+1])] += 1
    
    if not pair_counts:
        return tokens_list, None
    
    # Most frequent pair
    best_pair = pair_counts.most_common(1)[0][0]
    merged = best_pair[0] + best_pair[1]
    
    # Merge that pair everywhere
    new_tokens_list = []
    for tokens in tokens_list:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == best_pair:
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        new_tokens_list.append(new_tokens)
    
    return new_tokens_list, best_pair


# Demo
text = "the cat sat on the mat the cat"
tokens_list = [list(text)]  # start with characters

print(f"Initial: {tokens_list[0][:30]}... ({len(tokens_list[0])} tokens)")
print()

for step in range(10):
    tokens_list, pair = simple_bpe_step(tokens_list)
    if pair is None:
        break
    print(f"Step {step+1}: merge '{pair[0]}'+'{pair[1]}' → '{pair[0]+pair[1]}'  "
          f"→ {len(tokens_list[0])} tokens")

print(f"\nFinal tokens: {tokens_list[0]}")

---

<a id='hparams'></a>
## 4. Hiperparámetros de entrenamiento

Los LLMs se entrenan con configuraciones muy específicas:

### Learning rate

Se usa **warmup + cosine decay**:

1. **Warmup** (primeros ~2000 steps): el LR sube linealmente de 0 al LR máximo. Esto evita que el modelo haga pasos enormes al principio cuando los gradientes son ruidosos.

2. **Cosine decay**: después del warmup, el LR baja suavemente con forma de coseno hasta un LR mínimo (típicamente 10% del máximo).

### Gradient clipping

Se limita la norma global del gradiente (ej: `max_norm=1.0`). Si el gradiente es más grande, se escala proporcionalmente. Esto previene **exploding gradients** sin afectar la dirección.

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

### Weight decay

Regularización L2 en los parámetros. AdamW lo implementa correctamente (decoupled weight decay). Típicamente 0.1.

### Batch size

Se usan batch sizes muy grandes (ej: 4M tokens por step). Como no cabe todo en una GPU, se usa **gradient accumulation**: se procesan mini-batches, se acumulan los gradientes, y se actualiza cada N mini-batches.

### Configuración típica (estilo GPT-3/LLaMA)

| Hiperparámetro | Valor típico |
|----------------|-------------|
| LR máximo | 3e-4 a 6e-4 |
| LR mínimo | LR_max / 10 |
| Warmup steps | 2000 |
| Weight decay | 0.1 |
| Gradient clipping | 1.0 |
| Batch size (tokens) | 2M - 4M |
| Context length | 2048 - 8192+ |
| Optimizer | AdamW (β1=0.9, β2=0.95) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Warmup + cosine decay visualization
def get_lr(step, warmup_steps=2000, max_lr=6e-4, min_lr=6e-5, total_steps=100000):
    if step < warmup_steps:
        return max_lr * (step / warmup_steps)
    elif step > total_steps:
        return min_lr
    else:
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return min_lr + 0.5 * (max_lr - min_lr) * (1 + np.cos(np.pi * progress))


steps = np.arange(120000)
lrs = [get_lr(s) for s in steps]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, lrs, 'b-', lw=1.5)
ax.axvline(2000, color='red', linestyle='--', alpha=0.5, label='Fin warmup')
ax.axvline(100000, color='green', linestyle='--', alpha=0.5, label='Fin cosine decay')
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule: Warmup + Cosine Decay')
ax.legend()
plt.tight_layout()
plt.show()

---

<a id='scaling'></a>
## 5. Scaling laws

Uno de los descubrimientos más importantes en LLMs (Kaplan et al., 2020) es que la performance sigue **leyes de potencia** (power laws):

$$L \propto N^{-\alpha_N} \cdot D^{-\alpha_D} \cdot C^{-\alpha_C}$$

Donde:
- $L$ = loss (menor es mejor)
- $N$ = número de parámetros
- $D$ = tamaño del dataset (tokens)
- $C$ = compute (FLOPs)

### ¿Qué significa?

- **Más parámetros** → mejor performance (si hay suficientes datos)
- **Más datos** → mejor performance (si el modelo es suficientemente grande)
- **Más compute** → mejor performance
- Las mejoras son **predecibles** y siguen curvas suaves en escala log-log

### Chinchilla scaling (Hoffmann et al., 2022)

Descubrieron que la mayoría de los LLMs estaban **under-trained**: tenían muchos parámetros pero pocos datos. La regla de Chinchilla dice que el número óptimo de tokens es **~20x** el número de parámetros.

| Modelo | Parámetros | Tokens óptimos (Chinchilla) |
|--------|------------|-----------------------------|
| 1B | 20B tokens |
| 7B | 140B tokens |
| 70B | 1.4T tokens |

Modelos recientes como LLaMA 3 entrenan con **mucho más** datos que lo que sugiere Chinchilla (15T+ tokens para 8B params), porque es más barato hacer inferencia con un modelo más chico que está sobre-entrenado en datos.

---

<a id='posttraining'></a>
## 6. Lo que sigue: post-training

El modelo que sale del pre-training es un **base model**. Puede completar texto pero no sigue instrucciones bien, puede generar contenido inapropiado, y no es muy "conversacional".

Para convertirlo en un **chat model** (como ChatGPT), se necesita **post-training**:

### SFT (Supervised Fine-Tuning)

Se entrena con ejemplos de alta calidad de conversaciones: pares (instrucción, respuesta ideal). El modelo aprende a seguir instrucciones y responder en formato conversacional.

### RLHF (Reinforcement Learning from Human Feedback)

1. Humanos evalúan y rankean respuestas del modelo
2. Se entrena un **reward model** con esos rankings
3. Se usa RL (PPO) para optimizar el LLM según el reward model

### DPO (Direct Preference Optimization)

Una alternativa más simple a RLHF que no necesita un reward model separado. Optimiza directamente basado en pares de preferencias (respuesta buena vs respuesta mala).

### El pipeline completo

```
Pre-training (trillones de tokens, weeks)
    → Base model
    → SFT (miles de ejemplos, hours)
    → RLHF/DPO (preferencias humanas, hours-days)
    → Chat model listo para producción
```

---

<a id='resumen'></a>
## 7. Resumen de la serie completa

¡Felicitaciones por llegar hasta acá! Recorrimos un camino largo desde cero hasta LLMs. Acá va el mapa completo de lo que cubrimos:

| # | Notebook | Temas principales |
|---|----------|-------------------|
| 01 | **Aprendizaje Supervisado** | Modelo, loss, optimización, regresión lineal, MSE, gradient descent, clasificación, sigmoid, softmax, cross-entropy |
| 02 | **Perceptrón** | Función lineal, activaciones (step, sigmoid, ReLU), dot product, frontera de decisión, bias, XOR |
| 03 | **Shallow Networks** | Combinación de perceptrones, no-linealidad, regiones lineales, universal approximation theorem |
| 04 | **Deep Networks** | Composición de redes, eficiencia computacional, "folding" del espacio, notación matricial |
| 05 | **Loss Functions** | Maximum Likelihood, NLL, por qué MSE (Gaussiana) y CE (categórica) |
| 06 | **Backpropagation** | Regla de la cadena, forward/backward pass, autograd, grafo computacional |
| 07 | **Entrenamiento Práctico** | SGD, batches/epochs, momentum, Adam, inicialización (Xavier/He), BatchNorm, LayerNorm, residual connections |
| 08 | **Regularización** | Overfitting, L2/weight decay, early stopping, ensembling, dropout, data augmentation, transfer learning |
| 09 | **Secuencias: RNN/LSTM** | RNN, vanishing gradients, LSTM gates, diseño de arquitecturas, seq2seq, hidden state bottleneck |
| 10 | **Attention** | Bahdanau, dot product, self-attention, Q/K/V, múltiples capas |
| 11 | **Transformers** | Multi-head attention, causal mask, transformer block, positional encoding, encoder-decoder vs decoder-only |
| 12 | **GPT from Scratch** | Implementación completa, training loop, generación autoregresiva |
| 13 | **Eficiencia y GPUs** | GPU architecture, memory/compute bounds, scaling, Flash Attention, mixed precision |
| 14 | **LLM Pre-training** | Datasets, BPE, scaling laws, post-training (SFT, RLHF, DPO) |

---

### La idea general

Todo se reduce a esto:

1. **Definís una arquitectura** (cómo fluye la información) → Transformer
2. **Definís una loss** (qué querés minimizar) → Next-token prediction (cross-entropy)
3. **Entrenás con datos** (gradient descent + backprop) → Trillones de tokens
4. **Escalás** (más params, más datos, más compute) → Scaling laws

Y de eso emergen modelos que pueden conversar, razonar, programar, traducir, y mucho más.

---

**Esta fue la última notebook de la serie. ¡Felicitaciones por llegar hasta acá!** 🎉

Si querés seguir profundizando, algunos recursos recomendados:
- [Andrej Karpathy - Neural Networks: Zero to Hero](https://karpathy.ai/zero-to-hero.html)
- [Understanding Deep Learning (Prince)](https://udlbook.github.io/udlbook/)
- [The Scaling Book (JAX team)](https://jax-ml.github.io/scaling-book/)
- [Raschka - Build a LLM from Scratch](https://github.com/rasbt/LLMs-from-scratch)